In [1]:
from decimal import Decimal, ROUND_HALF_UP


def round_to_n_decimals(value: str, n: int) -> Decimal:
    d = Decimal(value)
    quantizer = Decimal('1.' + '0' * n) if n > 0 else Decimal('1')
    return d.quantize(quantizer, rounding=ROUND_HALF_UP)


def find_last_digit_position(value_str: str) -> int:
    if '.' in value_str:
        decimal_part = value_str.split('.')[1]
        return -len(decimal_part)
    else:
        stripped = value_str.rstrip('0')
        return len(value_str) - len(stripped)


def count_significant_digits(value_str: str) -> list:
    digits_only = value_str.replace('.', '').replace('-', '')
    stripped = digits_only.lstrip('0')
    return list(stripped)


def get_digit_positions(value_str: str) -> list:
    result = []
    if '.' in value_str:
        int_part, dec_part = value_str.split('.')
    else:
        int_part = value_str
        dec_part = ''
    int_part = int_part.lstrip('-')
    found_significant = False
    for i, ch in enumerate(int_part):
        pos = len(int_part) - 1 - i
        if ch != '0':
            found_significant = True
        if found_significant:
            result.append((ch, pos))
    for i, ch in enumerate(dec_part):
        pos = -(i + 1)
        if ch != '0':
            found_significant = True
        if found_significant:
            result.append((ch, pos))
    return result

In [2]:
print("=" * 65)
print("УПРАЖНЕНИЕ 1: Округление до 2, 3 и 4 знаков после запятой")
print("=" * 65)
numbers = ['3.009982', '24.00551', '21.161728']
for num_str in numbers:
    print(f"\nИсходное число: {num_str}")
    for n in [2, 3, 4]:
        rounded = round_to_n_decimals(num_str, n)
        print(f"  до {n} зн.: {rounded}")

УПРАЖНЕНИЕ 1: Округление до 2, 3 и 4 знаков после запятой

Исходное число: 3.009982
  до 2 зн.: 3.01
  до 3 зн.: 3.010
  до 4 зн.: 3.0100

Исходное число: 24.00551
  до 2 зн.: 24.01
  до 3 зн.: 24.006
  до 4 зн.: 24.0055

Исходное число: 21.161728
  до 2 зн.: 21.16
  до 3 зн.: 21.162
  до 4 зн.: 21.1617


In [3]:
print("=" * 65)
print("УПРАЖНЕНИЕ 2: Предельные абсолютные и относительные погрешности")
print("=" * 65)
print("Условие: все цифры верны в строгом смысле.")
print("Значит: Δ = 0.5 · (единица последнего разряда)")
numbers = ['36.7', '2.489', '31.010', '0.031']
for num_str in numbers:
    value = Decimal(num_str)
    pos = find_last_digit_position(num_str)
    delta = Decimal('0.5') * Decimal(10) ** pos
    relative = (delta / abs(value)) * 100
    print(f"\n  Число: {num_str}")
    print(f"  Последний разряд: 10^({pos}) = {Decimal(10) ** pos}")
    print(f"  Δ = 0.5 × 10^({pos}) = {delta}")
    print(f"  δ = {delta} / {value} × 100% ≈ {round(float(relative), 4)}%")

УПРАЖНЕНИЕ 2: Предельные абсолютные и относительные погрешности
Условие: все цифры верны в строгом смысле.
Значит: Δ = 0.5 · (единица последнего разряда)

  Число: 36.7
  Последний разряд: 10^(-1) = 0.1
  Δ = 0.5 × 10^(-1) = 0.05
  δ = 0.05 / 36.7 × 100% ≈ 0.1362%

  Число: 2.489
  Последний разряд: 10^(-3) = 0.001
  Δ = 0.5 × 10^(-3) = 0.0005
  δ = 0.0005 / 2.489 × 100% ≈ 0.0201%

  Число: 31.010
  Последний разряд: 10^(-3) = 0.001
  Δ = 0.5 × 10^(-3) = 0.0005
  δ = 0.0005 / 31.010 × 100% ≈ 0.0016%

  Число: 0.031
  Последний разряд: 10^(-3) = 0.001
  Δ = 0.5 × 10^(-3) = 0.0005
  δ = 0.0005 / 0.031 × 100% ≈ 1.6129%


In [4]:
print("=" * 65)
print("УПРАЖНЕНИЕ 3: Округление до сотых + верные цифры в строгом смысле")
print("=" * 65)
print("Условие: все цифры исходных чисел верны в строгом смысле.")
numbers = ['0.310', '3.495', '24.3790']
for num_str in numbers:
    value = Decimal(num_str)
    pos_orig = find_last_digit_position(num_str)
    delta_orig = Decimal('0.5') * Decimal(10) ** pos_orig
    rounded = round_to_n_decimals(num_str, 2)
    rounding_error = abs(value - rounded)
    total_error = delta_orig + rounding_error
    print(f"\n{'─' * 55}")
    print(f"  Исходное число:       a  = {num_str}")
    print(f"  Δ(a) [исходная]:      {delta_orig}")
    print(f"  Округлённое:          ã  = {rounded}")
    print(f"  |a − ã| [округление]: {rounding_error}")
    print(f"  Δ(ã) [полная]:        {delta_orig} + {rounding_error} = {total_error}")
    rounded_str = str(rounded)
    digit_positions = get_digit_positions(rounded_str)
    sig_digits = count_significant_digits(rounded_str)
    print(f"\n  Проверка верности значащих цифр ({rounded_str}):")
    print(f"  Значащие цифры: {', '.join(sig_digits)}")
    correct_count = 0
    for digit, pos in digit_positions:
        threshold = Decimal('0.5') * Decimal(10) ** pos
        is_correct = total_error <= threshold
        status = "верна" if is_correct else "неверна"
        print(f"    Цифра '{digit}' (разряд 10^{pos:+d}): "
              f"Δ(ã)={total_error} {'<=' if is_correct else '>'} "
              f"0.5·10^({pos})={threshold}  -> {status}")
        if is_correct:
            correct_count += 1
    print(f"\n  Верных значащих цифр в строгом смысле: {correct_count}")

УПРАЖНЕНИЕ 3: Округление до сотых + верные цифры в строгом смысле
Условие: все цифры исходных чисел верны в строгом смысле.

───────────────────────────────────────────────────────
  Исходное число:       a  = 0.310
  Δ(a) [исходная]:      0.0005
  Округлённое:          ã  = 0.31
  |a − ã| [округление]: 0.000
  Δ(ã) [полная]:        0.0005 + 0.000 = 0.0005

  Проверка верности значащих цифр (0.31):
  Значащие цифры: 3, 1
    Цифра '3' (разряд 10^-1): Δ(ã)=0.0005 <= 0.5·10^(-1)=0.05  -> верна
    Цифра '1' (разряд 10^-2): Δ(ã)=0.0005 <= 0.5·10^(-2)=0.005  -> верна

  Верных значащих цифр в строгом смысле: 2

───────────────────────────────────────────────────────
  Исходное число:       a  = 3.495
  Δ(a) [исходная]:      0.0005
  Округлённое:          ã  = 3.50
  |a − ã| [округление]: 0.005
  Δ(ã) [полная]:        0.0005 + 0.005 = 0.0055

  Проверка верности значащих цифр (3.50):
  Значащие цифры: 3, 5, 0
    Цифра '3' (разряд 10^+0): Δ(ã)=0.0055 <= 0.5·10^(0)=0.5  -> верна
    Цифра '5